<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; font-family: Arial, sans-serif;">

  <tr>
    <th colspan="2" style="text-align: left;">
      Toyota Corolla Car-Parts Classifier
    </th>
  </tr>

  <tr>
    <td colspan="2">
      Convolutional Neural Network for classifying 7 under-hood vehicle components from images.
    </td>
  </tr>

  <tr>
    <td><b>Dataset</b></td>
    <td>stevenalbert15 / toyota-corolla-car-parts (Kaggle)</td>
  </tr>

  <tr>
    <td><b>Total Images</b></td>
    <td>1,352 across 7 classes</td>
  </tr>

  <tr>
    <td><b>Data Split</b></td>
    <td>Train 80% · Validation 10% · Test 10%</td>
  </tr>

  <tr>
    <td><b>Hardware</b></td>
    <td>2 × NVIDIA Tesla T4 (13,757 MB each)</td>
  </tr>

  <tr>
    <td><b>Test Accuracy</b></td>
    <td>95.77%</td>
  </tr>

</table>

---
##  1 · Dataset Paths

Seven directories — one per part class — are mapped to Python variables.  
These paths point to the Kaggle input mount and are used later when loading the dataset.

|  Classes |
|---|
|  Air Filter |
|  Battery |
|  Coolant Reservoir |
|  Engine |
|  Engine Cover |
|  Reservoir Cap |
|  Windshield Fluid |

In [ ]:
Air_Filter = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Air Filter'
Battery = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Battery'
Coolant_Reservoir = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Coolant Reservoir'
Engine = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Engine'
Engine_Cover = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Engine Cover'
Reservoir_Cap = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Reservoir Cap'
Windshield_Fluid = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset/Windshield Fluid'

---
##  2 · Imports

`TensorFlow / Keras` is the sole deep-learning dependency.  
`os` is imported for any path utilities used later in the notebook.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

---
##  3 · Global Hyperparameters

| Parameter | Value | Rationale |
|---|---|---|
| `img_size` | `(224, 224)` | Standard input for CNN baselines; matches ImageNet-pretrained backbones |
| `batch_size` | `32` | Good balance between GPU memory usage and gradient stability |
| `seed` | `42` | Ensures reproducible train/val splits across runs |
| `base_dir` | Kaggle dataset root | Single source of truth for all `image_dataset_from_directory` calls |

In [ ]:
base_dir = '/kaggle/input/datasets/stevenalbert15/toyota-corolla-car-parts/Toyota Corolla Dataset'

img_size = (224, 224)
batch_size = 32
seed = 42

---
##  4 · Training Dataset

`image_dataset_from_directory` scans `base_dir`, infers class labels from sub-folder names, and splits **80 %** of images into the training set.

> **Result:** 1 082 training images across 7 classes, resized to 224 × 224 and batched to 32.

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

---
##  5 · Temporary Validation Pool

The remaining **20 %** (270 images) are loaded as a staging pool that will be split further into a proper validation set and a held-out test set in the next step.

In [ ]:
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

---
##  6 · Val / Test Split

The 270-image pool is divided **50 / 50**:

- `val_ds` → first half (~135 images) – used during training to monitor generalisation
- `test_ds` → second half (~135 images) – held out until final evaluation

In [ ]:
val_size = int(0.5 * len(temp_ds))

val_ds = temp_ds.take(val_size)
test_ds = temp_ds.skip(val_size)

---
##  7 · Pipeline Optimisation & Class Names

Three important things happen here:

1. **`train_ds` is reloaded** so class names can be extracted cleanly before any dataset transformations are applied.
2. **`class_names`** is captured — this list drives the final Dense layer size and any prediction decoding.
3. **`AUTOTUNE`** is applied to `val_ds` and `test_ds`: caching eliminates re-reading from disk; prefetching overlaps data loading with GPU compute.

```
Classes: ['Air Filter', 'Battery', 'Coolant Reservoir',
          'Engine', 'Engine Cover', 'Reservoir Cap', 'Windshield Fluid']
```

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

# ✅ GET CLASS NAMES HERE (before transformations)
class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

---
##  8 · Class Names Confirmed

A quick sanity check — prints the 7 class labels derived from the directory structure.  
`num_classes = 7` feeds directly into the output layer of the model below.

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

---
##  9 · CNN Architecture

A four-block convolutional backbone followed by a fully-connected head.

```
Input (224 × 224 × 3)
  └─ Rescaling (/255)          ← normalise pixels to [0, 1]
  └─ Conv2D 32 × (3×3) + ReLU
  └─ MaxPooling2D               → 111 × 111
  └─ Conv2D 64 × (3×3) + ReLU
  └─ MaxPooling2D               → 54 × 54
  └─ Conv2D 128 × (3×3) + ReLU
  └─ MaxPooling2D               → 26 × 26
  └─ Conv2D 256 × (3×3) + ReLU
  └─ MaxPooling2D               → 12 × 12
  └─ Flatten                    → 36 864
  └─ Dense 256 + ReLU
  └─ Dropout 0.5                ← regularisation
  └─ Dense 7 + Softmax          ← one logit per class
```

**Design choices**
- Doubling filter counts (32 → 64 → 128 → 256) progressively captures richer features.
- `Dropout(0.5)` halves active neurons during training, reducing overfitting on the relatively small dataset.
- `Softmax` outputs a proper probability distribution over the 7 classes.

>  Training was **interrupted** (KeyboardInterrupt) during this run, but the model weights saved below reflect a prior completed training session with 95.77 % test accuracy.

In [ ]:
model = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

---
##  10 · Test Set Evaluation

The model is evaluated on the fully held-out `test_ds` (~135 images it has never seen).

| Metric | Value |
|---|---|
| Test Loss | 0.1642 |
| **Test Accuracy** | **95.77 %** |

> Only ~5 images in 135 were misclassified — excellent performance for a custom CNN trained from scratch on ~1 000 images.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

---
##  11 · Model Export

The trained model is serialised to `car_parts_cnn.h5` (Keras HDF5 format).

>  **Tip:** Keras now recommends the native `.keras` format (`model.save('car_parts_cnn.keras')`) for better forward compatibility. HDF5 remains fully functional.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

---
##  12 · Data Augmentation Pipeline (Next Iteration)

Three geometric transforms are stacked into a preprocessing sub-model:

| Layer | Effect | Rationale |
|---|---|---|
| `RandomFlip("horizontal")` | Mirror images left-right | Parts look the same from either side |
| `RandomRotation(0.1)` | ±36° random rotation | Handles camera tilt / non-level placement |
| `RandomZoom(0.1)` | ±10 % zoom in/out | Simulates varying shooting distances |

These augmentations are **applied on-the-fly** during training (only), effectively multiplying the apparent dataset size and improving the model's robustness to real-world variation.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)

print("Test Accuracy:", test_acc)

---
##  13 · Augmented Model Skeleton

This cell sketches the **augmentation-first** model structure that replaces the plain `Rescaling` baseline above.  
The `...` placeholder marks where the Conv/Dense blocks from Cell 9 would be inserted.

> This is the recommended architecture for the **next training run** — combining augmentation + the proven 4-block CNN should push accuracy above 96 %.

In [ ]:
model.save("car_parts_cnn.h5")

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
model = keras.Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    ...
])

---
<div style=" border-left: 4px solid ; padding: 20px 24px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; ">

###  Summary

| Stage | Detail |
|---|---|
| **Task** | Multi-class image classification (7 Toyota Corolla parts) |
| **Model** | Custom CNN — 4 Conv blocks + Dense head |
| **Dataset** | 1 352 images, 80/10/10 train/val/test split |
| **Final accuracy** | 95.77 % on unseen test images |

</div>